## Treinamento por reforço Multi-agente

Compara ações **aleatórias** x **Q-Learning independente** x **Q-Learning independente com política compartilhada**

In [1]:
import random
import numpy as np
from immune_rl.graph import build_pilot_graph
from immune_rl.dynamics import SimpleDynamics, PathogenInflammationReward
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import RandomPolicy, QLearningPolicy
from immune_rl.encoding import StateEncoder

In [2]:
EPISODES = 1500

In [3]:
def make_env():
    return ImmuneEnvironment(build_pilot_graph, SimpleDynamics(), PathogenInflammationReward())

def run_episode(env, policies, train):
    obs = env.reset()
    team_return = 0.0
    while True:
        actions = {cid: policies[cid].select_action(o) for cid, o in obs.items()}
        next_obs, rewards, terminated, truncated, info = env.step(actions)
        if train:
            for cid in obs:
                policies[cid].update(obs[cid], actions[cid], rewards[cid], next_obs[cid], terminated)
        team_return += sum(rewards.values()) / len(rewards)
        obs = next_obs
        if terminated or truncated:
            if train:
                for p in set(policies.values()):   # set: política compartilhada decai 1x só
                    p.end_episode()
            return team_return
 
 
def experiment(name, build_policies, seed):
    random.seed(seed)
    env = make_env()
    cells = [c.id for c in env.graph.cells()]
    policies = build_policies(cells, seed)
    returns = [run_episode(env, policies, train=True) for _ in range(EPISODES)]
    for p in set(policies.values()):
        if hasattr(p, "epsilon"):
            p.epsilon = 0.0                        # avaliação de exploit
    evals = [run_episode(env, policies, train=False) for _ in range(200)]
    return np.mean(returns[:100]), np.mean(returns[-100:]), np.mean(evals)
 
 
def random_pols(cells, seed):
    return {c: RandomPolicy() for c in cells}
 
def independent(cells, seed):
    return {c: QLearningPolicy(StateEncoder(), seed=seed + i) for i, c in enumerate(cells)}
 
def shared(cells, seed):
    p = QLearningPolicy(StateEncoder(), seed=seed)
    return {c: p for c in cells}

In [4]:
print(f"{'configuração':<22}{'início':>9}{'fim treino':>12}{'avaliação':>11}   (retorno de equipe por episódio)")

for name, builder in [("aleatório", random_pols), ("IQL independente", independent), ("IQL compartilhado", shared)]:
        res = np.array([experiment(name, builder, seed) for seed in range(5)]).mean(axis=0)
        print(f"{name:<22}{res[0]:>9.3f}{res[1]:>12.3f}{res[2]:>11.3f}")

configuração             início  fim treino  avaliação   (retorno de equipe por episódio)
aleatório                 0.634       0.631      0.620
IQL independente          0.819       0.955      0.970
IQL compartilhado         0.815       0.980      0.991
